In [1]:
from __future__ import print_function
import argparse

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import TensorDataset, DataLoader
from torch.nn.parallel import DataParallel

import idx2numpy

import numpy as np

import tiledb
from tiledb.ml.models.pytorch import PyTorchTileDBModel



In [2]:

## DEFINE MODEL

class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.dropout1 = nn.Dropout(0.25)
        self.dropout2 = nn.Dropout(0.5)
        self.fc1 = nn.Linear(9216, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.conv2(x)
        x = F.relu(x)
        x = F.max_pool2d(x, 2)
        x = self.dropout1(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout2(x)
        x = self.fc2(x)
        output = F.log_softmax(x, dim=1)
        return output


def train(model, device, train_loader, optimizer, epoch):
    
    log_interval = 10
    
    model.train()
    dry_run = True
    for batch_idx, (data, target) in enumerate(train_loader):
        print("data: ", data.shape, data.dtype, " target: ", target.shape, target.dtype)
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = F.nll_loss(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx % log_interval == 0:
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, batch_idx * len(data), len(train_loader.dataset),
                100. * batch_idx / len(train_loader), loss.item()))
            if dry_run:
                break


def test(model, device, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += F.nll_loss(output, target, reduction='sum').item()  # sum up batch loss
            pred = output.argmax(dim=1, keepdim=True)  # get the index of the max log-probability
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)

    print('\nTest set: Average loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
        test_loss, correct, len(test_loader.dataset),
        100. * correct / len(test_loader.dataset)))


In [3]:

## SET PARAMETERS

use_cuda = True
use_mps = False

seed = 1

batch_size = 64
test_batch_size = 1000

lr = 1.0
gamma = 0.7

epochs = 14

torch.manual_seed(seed)

if use_cuda:
    device = torch.device("cuda")
elif use_mps:
    device = torch.device("mps")
else:
    device = torch.device("cpu")

train_kwargs = {'batch_size': batch_size}
test_kwargs = {'batch_size': test_batch_size}
if use_cuda:
    cuda_kwargs = {'num_workers': 1,
                   'pin_memory': True,
                   'shuffle': True}
    train_kwargs.update(cuda_kwargs)
    test_kwargs.update(cuda_kwargs)
    
transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])


In [4]:

## DOWNLOAD DATASET

dataset1 = datasets.MNIST('data', train=True, download=True,
                   transform=transform)

# dataset2 = datasets.MNIST('data', train=False,
#                    transform=transform)


100%|██████████| 9912422/9912422 [00:00<00:00, 95584697.62it/s]


Extracting data/MNIST/raw/train-images-idx3-ubyte.gz to data/MNIST/raw



100%|██████████| 28881/28881 [00:00<00:00, 111133664.06it/s]

Extracting data/MNIST/raw/train-labels-idx1-ubyte.gz to data/MNIST/raw



100%|██████████| 1648877/1648877 [00:00<00:00, 58726702.53it/s]


Extracting data/MNIST/raw/t10k-images-idx3-ubyte.gz to data/MNIST/raw



100%|██████████| 4542/4542 [00:00<00:00, 12118656.98it/s]

Extracting data/MNIST/raw/t10k-labels-idx1-ubyte.gz to data/MNIST/raw



In [5]:

## SAVE DATA TO TILEDB

file = 'data/MNIST/raw/train-images-idx3-ubyte'
train_data = idx2numpy.convert_from_file(file)
# print("data shape: ", train_data.shape)
file = 'data/MNIST/raw/train-labels-idx1-ubyte'
train_labels = idx2numpy.convert_from_file(file)
# print("labels: ", train_labels.shape)


try:
    tiledb.from_numpy("train_data", train_data)
    tiledb.from_numpy("train_labels", train_labels)
except:
    print("already exists")


with tiledb.open("train_data") as A:
    data = A[:]
with tiledb.open("train_labels") as A:
    labels = A[:]


print(data.shape)
print(labels.shape)



(60000, 28, 28)
(60000,)


In [6]:

## TRAIN MODEL

train_data = []
for i in range(len(data)):
   train_data.append([data[i][np.newaxis, :].astype(np.float32), labels[i]])

data_loader = torch.utils.data.DataLoader(train_data, **train_kwargs)

model = Net().to(device)

dp_model = DataParallel(model, device_ids=[0, 1, 2, 3])

optimizer = optim.Adadelta(dp_model.parameters(), lr=lr)
scheduler = StepLR(optimizer, step_size=1, gamma=gamma)

for epoch in range(1, epochs + 1):
    train(dp_model, device, data_loader, optimizer, epoch)
    scheduler.step()


data:  torch.Size([64, 1, 28, 28]) torch.float32  target:  torch.Size([64]) torch.uint8
Train Epoch: 1 [0/60000 (0%)]	Loss: 13.844046
data:  torch.Size([64, 1, 28, 28]) torch.float32  target:  torch.Size([64]) torch.uint8
Train Epoch: 2 [0/60000 (0%)]	Loss: 48.815102
data:  torch.Size([64, 1, 28, 28]) torch.float32  target:  torch.Size([64]) torch.uint8
Train Epoch: 3 [0/60000 (0%)]	Loss: 25.321886
data:  torch.Size([64, 1, 28, 28]) torch.float32  target:  torch.Size([64]) torch.uint8
Train Epoch: 4 [0/60000 (0%)]	Loss: 10.624513
data:  torch.Size([64, 1, 28, 28]) torch.float32  target:  torch.Size([64]) torch.uint8
Train Epoch: 5 [0/60000 (0%)]	Loss: 3.719455
data:  torch.Size([64, 1, 28, 28]) torch.float32  target:  torch.Size([64]) torch.uint8
Train Epoch: 6 [0/60000 (0%)]	Loss: 3.168170
data:  torch.Size([64, 1, 28, 28]) torch.float32  target:  torch.Size([64]) torch.uint8
Train Epoch: 7 [0/60000 (0%)]	Loss: 2.344037
data:  torch.Size([64, 1, 28, 28]) torch.float32  target:  torch.

In [7]:

## SAVE MODEL TO TILEDB


# uri = os.path.join(data_home, 'pytorch-mnist-1')
tiledb_model_1 = PyTorchTileDBModel(uri="pytorch-mnist-1", model=model, optimizer=optimizer)

tiledb_model_1.save(meta={'epochs': epochs})


